# Lezione 10 - Agenti AI in Produzione

In questa lezione imparerai i **modelli di produzione** per agenti AI utilizzando il Microsoft Agent Framework (Python). Tratteremo:

- **Osservabilità** — aggiunta di misurazione dei tempi e registrazione alle interazioni dell'agente
- **Valutazione** — utilizzo di un agente valutatore per punteggiare la qualità delle risposte
- **Gestione dei costi** — strategie per l'ottimizzazione dei token e la selezione del modello

Lo scenario è un **agente di viaggio** che aiuta gli utenti a pianificare viaggi, con monitoraggio e valutazione sovrapposti.


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considerazioni per la produzione

Passare dagli agenti AI da prototipi a produzione richiede una particolare attenzione a tre pilastri:

1. **Osservabilità** — Hai bisogno di visibilità su cosa sta facendo l'agente, quanto tempo impiega e quali strumenti utilizza. Senza tracciamento e logging è quasi impossibile eseguire il debug dei problemi in produzione.

2. **Valutazione** — Controlli di qualità automatizzati assicurano che le risposte dell’agente rimangano accurate, complete e utili nel tempo. Un agente valutatore può assegnare punteggi alle risposte in base a criteri definiti.

3. **Gestione dei costi** — L’uso dei token influisce direttamente sui costi. Strategie come l’ottimizzazione del prompt, la selezione del modello e la memorizzazione nella cache aiutano a mantenere le spese sotto controllo senza sacrificare la qualità.


## Costruire un agente osservabile

Definiamo gli strumenti di viaggio e avvolgiamo la chiamata all'agente con il timing in modo da poter monitorare la latenza. In produzione si integrerebbe con OpenTelemetry o un backend di tracciamento simile.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modelli di Valutazione

Un modello comune in produzione è utilizzare un secondo agente come **valutatore**. Il valutatore assegna un punteggio alla risposta dell'agente principale secondo criteri predefiniti come completezza, accuratezza e utilità.

Questo consente:
- Controlli automatici di qualità prima che le risposte raggiungano gli utenti
- Rilevamento di regressioni quando cambiano prompt o modelli
- Monitoraggio continuo delle prestazioni dell'agente nel tempo


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie di Gestione dei Costi

Controllare i costi è fondamentale per gli agenti di produzione AI. Ecco le strategie chiave:

| Strategia | Descrizione |
|---|---|
| **Ottimizzazione del prompt** | Mantieni le istruzioni di sistema concise. Rimuovi il contesto ridondante per ridurre i token di input. |
    "| **Selezione del modello** | Usa modelli più piccoli e meno costosi (es. GPT-5-mini) per compiti semplici come classificazione o estrazione, e riserva modelli più grandi per ragionamenti complessi. |\n",
| **Caching** | Memorizza i risultati degli strumenti e le query frequenti per evitare chiamate API ridondanti. |
| **Limiti di token** | Imposta limiti di `max_tokens` per evitare risposte inaspettatamente lunghe. |
| **Batching** | Raggruppa più query utente in una singola chiamata API quando possibile. |

In pratica, un approccio a livelli funziona bene: indirizza richieste semplici a un modello veloce ed economico e scala solo le query complesse a uno più potente.


## Riepilogo

In questa lezione hai imparato a:

1. **Aggiungere osservabilità** alle interazioni degli agenti con tempistiche e registrazioni, ponendo le basi per il tracciamento e il monitoraggio.
2. **Valutare automaticamente le risposte degli agenti** utilizzando un agente valutatore che assegna punteggi a completezza, accuratezza e utilità.
3. **Gestire i costi** attraverso l’ottimizzazione dei prompt, la selezione del modello, la memorizzazione nella cache e i budget di token.

Questi modelli di produzione aiutano a garantire che i tuoi agenti AI siano affidabili, misurabili e convenienti su larga scala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
